In [ ]:
import pandas as pd
import os

In [ ]:
SCAN_RESULTS_PATH = "../data/scan_results"
scan_results = os.listdir(SCAN_RESULTS_PATH)
scan_results

In [ ]:
df = pd.concat(
    [
        pd.read_csv(f"{SCAN_RESULTS_PATH}/{filename}") for filename in scan_results
    ]
)
df

In [ ]:
df["tp"] = df["malicious"] & df["blocked"]
df["fp"] = ~df["malicious"] & df["blocked"]
df["tn"] = ~df["malicious"] & ~df["blocked"]
df["fn"] = df["malicious"] & ~df["blocked"]
df

In [ ]:
import json
with open("../data/dataset_full.json", "r", encoding="utf8") as file:
    dataset = json.load(file)

dataset[:2]

In [ ]:
malicious_return = [
    scenario["id"]
    for scenario in dataset
    if any(
        tool["return_value"] != ""
        for tool in scenario["server"]["tools"]
    )
]

malicious_return

In [ ]:
df["malicious_return"] = df["scenario_id"].isin(malicious_return)
df

## Evaluation

In [ ]:
g = (
    df[df["malicious_return"] == False]
    .drop_duplicates(subset=["scanner", "name", "description"], keep="first")
    .groupby("scanner")[["tp", "fp", "fn"]]
    .sum()
)
g

In [ ]:
precision = g["tp"] / (g["tp"] + g["fp"])
recall = g["tp"] / (g["tp"] + g["fn"])
f1 = 2 * (precision * recall) / (precision + recall)

metrics = pd.DataFrame({"precision": precision, "recall": recall, "f1": f1})
metrics.sort_values(by="f1")

## Performance

Latency (ms):

In [ ]:
performance_metrics = df.groupby("scanner")["latency_ms"].agg(
    avg="mean",
    minimum="min",
    maximum="max",
    median="median",
    std="std",
    count="count",
)
performance_metrics.sort_values(by="median") # type: ignore